# Figure 3 End-to-End Recreation

This notebook is a minimal end-to-end recreation of the Figure 3 paired primary-versus-metastasis panels using the same shared clinical metadata file as `figure_1_scripts.ipynb`, together with the staged VCF, ClinCNV, OncoKB, BED, and cytoband inputs.


In [ ]:
from __future__ import annotations

import glob
import gzip
import os
import re
from collections import defaultdict
from pathlib import Path
from typing import Any

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from scipy.stats import fisher_exact, mannwhitneyu
from statsmodels.stats.multitest import multipletests

DEFAULT_NOTEBOOK_DIR = Path("/mnt/myvolume/PanelSeqMelanomaNotebook/analysis")
NOTEBOOK_DIR = Path(os.environ.get("PANELSEQ_NOTEBOOK_DIR", DEFAULT_NOTEBOOK_DIR)).expanduser().resolve()
REPO_ROOT = NOTEBOOK_DIR.parent
PANEL_SEQ_ROOT = Path(os.environ.get("PANELSEQ_DATA_ROOT", "/mnt/myvolume/panel_seq")).expanduser().resolve()

CLINCNV_RUN_DIR = PANEL_SEQ_ROOT / "v4_v5_clincnv/new_clincnv_runs/combined_runs/combined_bed_split_s100_l3_f1"
CLINCNV_SOMATIC_DIR = CLINCNV_RUN_DIR / "somatic"
V4_BED_PATH = PANEL_SEQ_ROOT / "v4_v5_clincnv/ssSC_v4.bed"
V5_BED_PATH = PANEL_SEQ_ROOT / "v4_v5_clincnv/ssSC_v5.bed"
CYTOBAND_PATH = PANEL_SEQ_ROOT / "gistic/hg38_cytoBandIdeo.txt"
VCF_DIR = PANEL_SEQ_ROOT / "new_bed_analysis/vcfs"
ONCOKB_DIR = PANEL_SEQ_ROOT / "new_bed_analysis/oncokb/oncokb_results_pass_vaf002_alt5"

CLINICAL_METADATA_PATH = Path(
    os.environ.get(
        "PANELSEQ_FIG1_CLINICAL_PATH",
        str(PANEL_SEQ_ROOT / "figure_inputs/figure1_clinical_minimal.tsv"),
    )
).expanduser().resolve()


LOSS_GENES_SIX = ["CDKN2A", "CDKN2B", "TP53BP1"]
GAIN_GENES_SIX = ["CDK4", "CD276", "MCL1"]
BOX_GAIN_GENES = GAIN_GENES_SIX.copy()
BOX_LOSS_GENES = LOSS_GENES_SIX.copy()
BOX_DIRECTIONAL_GENES = list(dict.fromkeys(BOX_GAIN_GENES + BOX_LOSS_GENES))
AUTOSOMES = {str(i) for i in range(1, 23)}
MIN_SHARED_BP_PER_CYTOBAND = 1000
MIN_EVENT_SAMPLES = 3
MIN_NONEVENT_SAMPLES = 3
MAX_REGION_BAND_GAP = 2
FDR_SIG_THRESHOLD = 0.25
GENE_CALLOUTS = {
("AMP", "12q12-q15"): "CDK4",
("AMP", "1q21-q25"): "MCL1",
("AMP", "15q21-q26"): "CD276",
("HOMDEL", "9p21-p24"): "CDKN2A/B",
("HOMDEL", "9q21-p24"): "CDKN2A/B",
("HOMDEL", "15q14-q15"): "TP53BP1",
}
FILENAME_RE = re.compile(r"(?:(?:CNA_)?CNAs?|Annotated_ClinCNV)_(?P<sample>[^/\\]+)\.txt$", re.IGNORECASE)
CYTOBAND_RE = re.compile(r"^(?P<chrom>\d+)(?P<arm>[pq])(?P<band>\d+)$", re.IGNORECASE)
INVALID_GENES = {"0", ".", ""}
BASES = set("ACGT")
VCF_GLOB = ["**/filtered.*.annotated.vcf", "**/filtered.*.annotated.vcf.gz"]

plt.rcParams.update({
"figure.dpi": 200,
"savefig.dpi": 300,
"pdf.fonttype": 42,
"ps.fonttype": 42,
"font.family": "Liberation Sans",
"font.size": 12,
"axes.spines.top": False,
"axes.spines.right": False,
})


def glob_count(pattern):
    return len(glob.glob(str(pattern), recursive=True))


def path_count(path_like):
    return int(Path(path_like).exists())




















REQUIRED_FIGURE2_CLINICAL_COLUMNS = [
    "Sample",
    "patient_id",
    "relapse_any",
    "rfs_time_months",
]


def load_figure2_clinical_table(path=CLINICAL_METADATA_PATH):
    frame = pd.read_csv(path, sep="	")
    missing = [column for column in REQUIRED_FIGURE2_CLINICAL_COLUMNS if column not in frame.columns]
    if missing:
        raise ValueError(f"{path} is missing required Figure 2 clinical columns: {missing}")

    final_df = frame[REQUIRED_FIGURE2_CLINICAL_COLUMNS].copy().set_index("Sample").sort_index()
    if final_df.index.duplicated().any():
        duplicates = sorted(final_df.index[final_df.index.duplicated()].unique())
        raise ValueError(f"Figure 2 clinical input contains duplicated Sample IDs: {duplicates}")

    final_df["patient_id"] = final_df["patient_id"].astype(str).str.strip()
    final_df["relapse_any"] = pd.to_numeric(final_df["relapse_any"], errors="coerce").astype("Int64")
    final_df["rfs_time_months"] = pd.to_numeric(final_df["rfs_time_months"], errors="coerce")
    return final_df


def open_maybe_gzip(path: str | Path):
    path = str(path)
    return gzip.open(path, "rt") if path.endswith(".gz") else open(path, "rt")


def sample_name_from_path(path: str | Path) -> str:
    base = os.path.basename(str(path))
    if base.startswith("filtered."):
        base = base[len("filtered."):]
    for suffix in (".annotated.vcf.gz", ".annotated.vcf", ".vcf.gz", ".vcf"):
        if base.endswith(suffix):
            return base[: -len(suffix)]
    return base


def normalize_sample_id(value: Any) -> str:
    return str(value).split("__")[0].strip()


def parse_info_dict(info: Any) -> dict[str, Any]:
    parsed = {}
    for item in str(info or "").split(";"):
        if not item:
            continue
        if "=" in item:
            key, value = item.split("=", 1)
            parsed[key] = value
        else:
            parsed[item] = True
    return parsed


def looks_somatic(info: Any, strict: bool = False) -> bool:
    if not strict:
        return True
    markers = ["SOMATIC", "SOMATIC=1", "SS=2", "SSTATUS=Somatic", "TYPE=somatic", "SOMATIC;"]
    info_text = str(info or "")
    return any(marker in info_text for marker in markers)


def is_snv(ref: str, alt: str) -> bool:
    return len(ref) == 1 and len(alt) == 1 and ref in BASES and alt in BASES


def canon_chrom(value: Any) -> str:
    if value is None:
        return ""
    return re.sub(r"^chr", "", str(value).strip(), flags=re.IGNORECASE)


def get_vaf_dp(format_keys, sample_values, alt_index, info_dict=None):
    if not format_keys or not sample_values:
        vaf = None
        if info_dict and "AF" in info_dict:
            af_values = str(info_dict["AF"]).split(",")
            if alt_index < len(af_values):
                try:
                    vaf = float(af_values[alt_index])
                except Exception:
                    vaf = None
        return vaf, None

    fmt = dict(zip(format_keys, sample_values))
    dp = None
    if "DP" in fmt and fmt["DP"] not in (None, ".", ""):
        try:
            dp = int(float(fmt["DP"]))
        except Exception:
            dp = None

    vaf = None
    if "AF" in fmt and fmt["AF"] not in (None, ".", ""):
        af_values = str(fmt["AF"]).split(",")
        if alt_index < len(af_values):
            try:
                vaf = float(af_values[alt_index])
            except Exception:
                vaf = None

    if vaf is None and "AD" in fmt and fmt["AD"] not in (None, ".", ""):
        try:
            ad_values = [int(x) for x in str(fmt["AD"]).split(",")]
            if len(ad_values) >= 2 and (alt_index + 1) < len(ad_values):
                ref_count = ad_values[0]
                alt_count = ad_values[alt_index + 1]
                denom = ref_count + alt_count
                vaf = (alt_count / denom) if denom > 0 else 0.0
                if dp is None:
                    dp = int(sum(ad_values))
        except Exception:
            pass

    if vaf is None and info_dict and "AF" in info_dict:
        af_values = str(info_dict["AF"]).split(",")
        if alt_index < len(af_values):
            try:
                vaf = float(af_values[alt_index])
            except Exception:
                vaf = None

    return vaf, dp


def _genes_list(cell: Any) -> list[str]:
    if not cell or str(cell).upper() == "NA":
        return []
    return [gene.strip() for gene in str(cell).split(",") if gene.strip()]


def _looks_like_gene_symbol(value: Any) -> bool:
    token = str(value).strip().upper()
    return bool(token) and token not in INVALID_GENES and re.search(r"[A-Z]", token) is not None


def clean_gene_list(values: Any) -> list[str]:
    seen = set()
    cleaned = []
    for gene in values or []:
        gene_upper = str(gene).strip().upper()
        if not _looks_like_gene_symbol(gene_upper) or gene_upper in seen:
            continue
        seen.add(gene_upper)
        cleaned.append(gene_upper)
    return cleaned


def clean_gene_name(value: Any) -> str:
    gene = str(value).strip().upper()
    return gene if _looks_like_gene_symbol(gene) else ""


def sample_from_clincnv_filename(path: str | Path) -> str:
    match = re.search(
        r"(?:Annotated_CNA_CNAs?|Annotated_ClinCNV|CNAs)_(?P<pair_name>QHPRG[^.]+)\.txt$"
        r"|.+?_(?P<sample_suffix>QHPRG[^_]+)_Annotated_ClinCNV\.txt$",
        Path(path).name,
        re.IGNORECASE,
    )
    if not match:
        return Path(path).stem
    return match.group("pair_name") or match.group("sample_suffix")


def iter_clincnv_paths(root: str | Path):
    root = Path(root)
    manifest_path = root / "aneuploidy_manifest.tsv"
    if manifest_path.is_file():
        manifest_df = pd.read_csv(manifest_path, sep="      ", dtype=str).fillna("")
        seen = set()
        manifest_paths = []
        for raw_path in manifest_df.get("source_file", pd.Series(dtype=str)):
            if not raw_path:
                continue
            path = Path(raw_path)
            if not path.is_absolute():
                path = (root / raw_path).resolve()
            if not path.is_file() or path in seen:
                continue
            if not re.search(r"QHPRG", path.name, re.IGNORECASE):
                continue
            seen.add(path)
            manifest_paths.append(path)
        if manifest_paths:
            for path in manifest_paths:
                yield path
            return

    patterns = [
        "somatic/*/Annotated_ClinCNV_*.txt",
        "somatic/*/Annotated_CNA_CNAs_*.txt",
        "**/Annotated_ClinCNV_*.txt",
        "**/Annotated_CNA_CNAs_*.txt",
        "somatic/*/CNAs_*.txt",
        "**/CNAs_*.txt",
    ]
    seen = set()
    for pattern in patterns:
        for path in sorted(root.glob(pattern)):
            if path.is_file() and path not in seen and re.search(r"QHPRG", path.name, re.IGNORECASE):
                seen.add(path)
                yield path


def _coerce_float(value: Any) -> float | None:
    try:
        return float(value)
    except Exception:
        return None


def _coerce_int_like(value: Any) -> int | None:
    try:
        return int(float(value))
    except Exception:
        return None


def _state_to_bucket(state: Any) -> str | None:
    state = str(state).strip().upper()
    if state == "AMP":
        return "amp"
    if state == "GAIN":
        return "gained"
    if state == "DEL":
        return "lost"
    if state == "LOH":
        return "loh"
    return None


def _read_table(path: Path) -> list[dict[str, str]]:
    rows = []
    header_cols = None
    with open(path, "r", encoding="utf-8") as handle:
        for line in handle:
            if line.startswith("##"):
                continue
            if line.startswith("#"):
                header_cols = [column.strip() for column in line[1:].rstrip("\n").split("\t")]
                continue
            if not line.strip() or header_cols is None:
                continue
            values = [value.strip() for value in line.rstrip("\n").split("\t")]
            if len(values) < len(header_cols):
                values += [""] * (len(header_cols) - len(values))
            rows.append(dict(zip(header_cols, values[: len(header_cols)])))
    return rows


def _parse_header(path: Path) -> dict[str, Any]:
    header = {}
    with open(path, "r", encoding="utf-8") as handle:
        for line in handle:
            if not line.startswith("##"):
                if line.startswith("#"):
                    break
                continue
            raw = line[2:].strip()
            if ":" in raw:
                key, value = raw.split(":", 1)
            elif "=" in raw:
                key, value = raw.split("=", 1)
            else:
                continue
            header[key.strip().lower()] = value.strip()
    return header


def _event_chrom(event: dict[str, str]) -> str | None:
    for key in event.keys():
        if key.lower() in {"chr", "chrom", "chromosome"}:
            raw = str(event[key]).strip()
            if not raw:
                return None
            return re.sub(r"^chr", "", raw, flags=re.IGNORECASE).upper()
    return None


def _classify_del(event: dict[str, str]) -> str:
    for major_key, minor_key in [("major_CN_allele", "minor_CN_allele"), ("major_CN_allele2", "minor_CN_allele2")]:
        major = _coerce_int_like(event.get(major_key))
        minor = _coerce_int_like(event.get(minor_key))
        if major == 0 and minor == 0:
            return "HOMDEL"
    return "HETDEL"


def summarize_sample(path: Path) -> dict[str, Any]:
    header = _parse_header(path)
    sample = header.get("combined_tumor_sample") or sample_from_clincnv_filename(path).split("-")[0]

    genes_amp, genes_gained = set(), set()
    genes_het_del, genes_hom_del, genes_loh = set(), set(), set()
    amp_like_cn_by_gene = {}

    for event in _read_table(path):
        bucket = _state_to_bucket(event.get("state"))
        if bucket is None:
            continue

        chrom = _event_chrom(event)
        genes = clean_gene_list(_genes_list(event.get("genes")))
        total_cn = _coerce_float(event.get("tumor_CN_change"))

        if bucket in {"amp", "gained"}:
            target_genes = genes_amp if bucket == "amp" else genes_gained
            target_genes.update(genes)
            for gene in genes:
                if total_cn is None:
                    continue
                amp_like_cn_by_gene[gene] = max(total_cn, amp_like_cn_by_gene.get(gene, float("-inf")))
        elif bucket == "lost":
            if chrom == "Y":
                continue
            if _classify_del(event) == "HOMDEL":
                genes_hom_del.update(genes)
            else:
                genes_het_del.update(genes)
        elif bucket == "loh":
            if chrom in {"X", "Y"}:
                continue
            genes_loh.update(genes)

    amp_like_cn_by_gene = {gene: cn for gene, cn in sorted(amp_like_cn_by_gene.items()) if cn != float("-inf")}
    return {
        "sample": sample,
        "genes_amp": sorted(genes_amp),
        "genes_gained": sorted(genes_gained),
        "genes_het_del": sorted(genes_het_del),
        "genes_hom_del": sorted(genes_hom_del),
        "genes_loh": sorted(genes_loh),
        "amp_like_cn_by_gene": amp_like_cn_by_gene,
    }


def merge_gene_lists(series: pd.Series) -> list[str]:
    seen = set()
    merged = []
    for values in series:
        for gene in values or []:
            gene_upper = str(gene).strip().upper()
            if gene_upper in INVALID_GENES or not gene_upper or gene_upper in seen:
                continue
            seen.add(gene_upper)
            merged.append(gene_upper)
    return merged


def merge_max_dicts(series: pd.Series) -> dict[str, float]:
    merged = {}
    for value in series:
        if not isinstance(value, dict):
            continue
        for gene, cn in value.items():
            if cn is None:
                continue
            gene_upper = str(gene).strip().upper()
            merged[gene_upper] = max(float(cn), merged.get(gene_upper, float("-inf")))
    return {gene: cn for gene, cn in sorted(merged.items()) if cn != float("-inf")}


def split_genes(value: Any) -> set[str]:
    if pd.isna(value) or value == "":
        return set()
    return {gene.strip().upper() for gene in str(value).split(",") if gene.strip()}


def detect_oncokb_variant_columns(df: pd.DataFrame):
    required = ["Chromosome", "Start_Position", "Reference_Allele", "Tumor_Seq_Allele2"]
    if all(column in df.columns for column in required):
        return ("Chromosome", "Start_Position", "Reference_Allele", "Tumor_Seq_Allele2")
    return None


def rowwise_key_membership(df, chrom_col, pos_col, ref_col, alt_col, vset):
    tmp = df[[chrom_col, pos_col, ref_col, alt_col]].copy()
    tmp["chrom"] = tmp[chrom_col].map(canon_chrom)
    tmp["pos"] = pd.to_numeric(tmp[pos_col], errors="coerce")
    tmp["ref"] = tmp[ref_col].astype(str)
    tmp["alt"] = tmp[alt_col].astype(str)

    def in_vcf(row):
        if pd.isna(row["pos"]):
            return False
        alts = [alt.strip() for alt in str(row["alt"]).split(",") if alt.strip() and alt.strip() != "."]
        return any(f"{row['chrom']}:{int(row['pos'])}:{row['ref']}:{alt}" in vset for alt in alts)

    return tmp.apply(in_vcf, axis=1)


def oncokb_sid_from_file(path: str | Path) -> str:
    base = os.path.basename(str(path))
    stem = os.path.splitext(base)[0]
    if stem.endswith(".oncokb"):
        stem = stem[: -len(".oncokb")]
    if stem.startswith("filtered."):
        stem = stem[len("filtered."):]
    return normalize_sample_id(stem)


def count_items(value: Any) -> int:
    if not isinstance(value, str) or not value.strip():
        return 0
    return sum(1 for item in value.split(",") if item.strip())


def extract_genes_from_label_list(label_str: Any) -> str:
    if not isinstance(label_str, str) or not label_str.strip():
        return ""
    genes = []
    for item in [x.strip() for x in label_str.split(",") if x.strip()]:
        match = re.match(r"^([A-Za-z0-9]+)", item)
        if match:
            genes.append(match.group(1))
    return ",".join(sorted(set(genes)))


def build_clincnv_summary(root_dir: str | Path) -> pd.DataFrame:
    root_dir = Path(root_dir)
    records = [summarize_sample(path) for path in iter_clincnv_paths(root_dir)]
    raw_df = pd.DataFrame(records)
    if raw_df.empty:
        return pd.DataFrame(columns=[
            "genes_amp", "genes_gained", "genes_het_del", "genes_hom_del", "genes_loh",
            "amp_like_cn_by_gene", "genes_amp_str", "genes_gained_str", "genes_HETDEL_str",
            "genes_HOMDEL_str", "genes_LOH_str",
        ])

    grouped = raw_df.groupby("sample", as_index=True).agg({
        "genes_amp": merge_gene_lists,
        "genes_gained": merge_gene_lists,
        "genes_het_del": merge_gene_lists,
        "genes_hom_del": merge_gene_lists,
        "genes_loh": merge_gene_lists,
        "amp_like_cn_by_gene": merge_max_dicts,
    })
    grouped["genes_amp_str"] = grouped["genes_amp"].apply(lambda values: ", ".join(values))
    grouped["genes_gained_str"] = grouped["genes_gained"].apply(lambda values: ", ".join(values))
    grouped["genes_HETDEL_str"] = grouped["genes_het_del"].apply(lambda values: ", ".join(values))
    grouped["genes_HOMDEL_str"] = grouped["genes_hom_del"].apply(lambda values: ", ".join(values))
    grouped["genes_LOH_str"] = grouped["genes_loh"].apply(lambda values: ", ".join(values))
    return grouped


def build_oncokb_summary(oncokb_dir: str | Path, vcf_dir: str | Path, *, min_vaf=0.0, strict_somatic=False, snv_only=False, include_samples=None) -> pd.DataFrame:
    vcf_paths = set()
    for pattern in VCF_GLOB:
        vcf_paths |= set(glob.glob(str(Path(vcf_dir) / pattern), recursive=True))

    sample_to_vcf = {}
    for path in sorted(vcf_paths):
        sid = normalize_sample_id(sample_name_from_path(path))
        sample_to_vcf.setdefault(sid, path)

    vcf_keyset = {}
    for sid, vcf_path in sample_to_vcf.items():
        keys = set()
        with open_maybe_gzip(vcf_path) as handle:
            for line in handle:
                if not line or line.startswith("#"):
                    continue
                cols = line.rstrip("\n").split("\t")
                if len(cols) < 10:
                    continue
                chrom, pos, _id, ref, alts, qual, flt, info = cols[:8]
                if flt != "PASS" or not looks_somatic(info, strict=strict_somatic):
                    continue
                info_dict = parse_info_dict(info)
                format_keys = cols[8].split(":")
                sample_values = cols[9].split(":")
                for alt_index, alt in enumerate(alts.split(",")):
                    if snv_only and not is_snv(ref, alt):
                        continue
                    vaf, _ = get_vaf_dp(format_keys, sample_values, alt_index, info_dict=info_dict)
                    if vaf is None or float(vaf) < float(min_vaf):
                        continue
                    keys.add(f"{canon_chrom(chrom)}:{int(pos)}:{ref}:{alt}")
        vcf_keyset[sid] = keys

    summary_rows = []
    for path in sorted(Path(oncokb_dir).glob("*.oncokb.tsv")):
        df = pd.read_csv(path, sep="\t", dtype=str).fillna("")
        sid = oncokb_sid_from_file(path)
        vcf_keys = vcf_keyset.get(sid, set())
        variant_cols = detect_oncokb_variant_columns(df)
        filtered_df = df if variant_cols is None else df.loc[rowwise_key_membership(df, *variant_cols, vcf_keys)].copy()

        mutation_effect = filtered_df.get("MUTATION_EFFECT", pd.Series("", index=filtered_df.index)).astype(str)
        oncogenic = filtered_df.get("ONCOGENIC", pd.Series("", index=filtered_df.index)).astype(str)
        hugo_symbol = filtered_df.get("ONCOKB_HUGO_SYMBOL", pd.Series("", index=filtered_df.index)).astype(str)
        protein_change = filtered_df.get("ONCOKB_PROTEIN_CHANGE", pd.Series("", index=filtered_df.index)).astype(str)

        gof_mask = mutation_effect.str.contains("Gain-of-function", case=False, na=False)
        lof_mask = mutation_effect.str.contains("Loss-of-function", case=False, na=False)
        oncogenic_mask = oncogenic.str.contains("Oncogenic", case=False, na=False)
        filtered_df["ONCOKB_HUGO_SYMBOL_CLEAN"] = hugo_symbol.map(clean_gene_name)
        valid_gene_mask = filtered_df["ONCOKB_HUGO_SYMBOL_CLEAN"].ne("")
        filtered_df["variant_label"] = (filtered_df["ONCOKB_HUGO_SYMBOL_CLEAN"].astype(str) + " " + protein_change.astype(str)).str.strip()

        gof_variants = filtered_df.loc[valid_gene_mask & gof_mask & oncogenic_mask, "variant_label"].dropna().unique().tolist()
        lof_one_allele = filtered_df.loc[valid_gene_mask & lof_mask & oncogenic_mask, "variant_label"].dropna().unique().tolist()
        lof_counts = filtered_df.loc[valid_gene_mask & lof_mask & oncogenic_mask, "ONCOKB_HUGO_SYMBOL_CLEAN"].value_counts()
        lof_biallelic = lof_counts[lof_counts > 1].index.tolist()

        summary_rows.append({
            "VCF_sid_used": sid,
            "GOF_Oncogenic_Variants": ", ".join(gof_variants) if gof_variants else "",
            "LOF_Oncogenic_One_Allele": ", ".join(lof_one_allele) if lof_one_allele else "",
            "LOF_Oncogenic_Both_Alleles": ", ".join(lof_biallelic) if lof_biallelic else "",
        })

    summary_df = pd.DataFrame(summary_rows)
    for column in ["GOF_Oncogenic_Variants", "LOF_Oncogenic_One_Allele", "LOF_Oncogenic_Both_Alleles"]:
        summary_df[column] = summary_df[column].fillna("").astype(str)
    summary_df["GOF_Genes"] = summary_df["GOF_Oncogenic_Variants"].map(extract_genes_from_label_list)
    summary_df["LOF_OneAllele_Genes"] = summary_df["LOF_Oncogenic_One_Allele"].map(extract_genes_from_label_list)
    summary_df["LOF_Biallelic_Genes"] = summary_df["LOF_Oncogenic_Both_Alleles"].map(extract_genes_from_label_list)
    summary_df["Variant_Richness"] = (
        summary_df["GOF_Oncogenic_Variants"].map(count_items)
        + summary_df["LOF_Oncogenic_One_Allele"].map(count_items)
        + summary_df["LOF_Oncogenic_Both_Alleles"].map(count_items)
    )
    summary_df = summary_df.sort_values(["VCF_sid_used", "Variant_Richness"], ascending=[True, False]).drop_duplicates("VCF_sid_used", keep="first").set_index("VCF_sid_used")

    if include_samples is not None:
        include_index = pd.Index([normalize_sample_id(sample) for sample in include_samples], name="VCF_sid_used").drop_duplicates()
        summary_df = summary_df.reindex(include_index)
        for column in ["GOF_Oncogenic_Variants", "LOF_Oncogenic_One_Allele", "LOF_Oncogenic_Both_Alleles", "GOF_Genes", "LOF_OneAllele_Genes", "LOF_Biallelic_Genes"]:
            if column in summary_df.columns:
                summary_df[column] = summary_df[column].fillna("").astype(str)
        if "Variant_Richness" in summary_df.columns:
            summary_df["Variant_Richness"] = summary_df["Variant_Richness"].fillna(0).astype(int)
    return summary_df


def add_functional_biallelic_calls(sample_df: pd.DataFrame, include_loh_with_lof: bool = True) -> pd.DataFrame:
    sample_df = sample_df.copy()
    rows = []
    for _, row in sample_df.iterrows():
        het = split_genes(row.get("genes_HETDEL_str"))
        loh = split_genes(row.get("genes_LOH_str"))
        hom = split_genes(row.get("genes_HOMDEL_str"))
        lof_one = split_genes(row.get("LOF_OneAllele_Genes"))
        lof_reported_biallelic = split_genes(row.get("LOF_Biallelic_Genes"))
        lof_any = lof_one | lof_reported_biallelic

        functional = set(hom) | (het & lof_any)
        if include_loh_with_lof:
            functional |= loh & lof_any
        functional = sorted(functional)

        rows.append({
            "functional_biallelic_genes": ",".join(functional) if functional else "",
            "functional_biallelic_loss": int(bool(functional)),
            "functional_biallelic_loss_n_genes": len(functional),
        })
    return sample_df.join(pd.DataFrame(rows, index=sample_df.index))


def compute_oncocycle(row: pd.Series, loss_set: set[str], gain_set: set[str]) -> int:
    functional_biallelic = split_genes(row.get("functional_biallelic_genes", ""))
    amps = split_genes(row.get("genes_amp_str", ""))
    gained = split_genes(row.get("genes_gained_str", ""))
    has_biallelic = bool(loss_set & functional_biallelic)
    has_gain = bool(gain_set & (amps | gained))
    return int(has_biallelic or has_gain)


V4_GENE_BED_PATH = PANEL_SEQ_ROOT / "v4_v5_clincnv/ssSC_v4.gc.genes.bedcoverage.sorted.bed"
V5_GENE_BED_PATH = PANEL_SEQ_ROOT / "v4_v5_clincnv/ssSC_v5.gc.genes.bedcoverage.sorted.bed"

PAIR_MET_SITE_ORDER = ["cutaneous / subcutaneous", "lymph node", "visceral", "Unknown"]
FIG3_REQUIRED_CLINICAL_COLUMNS = [
    "Sample",
    "patient_id",
    "panel_version",
    "paired_met_code",
    "paired_met_panel_version",
    "met_site_simple",
    "figure3_paired_primary",
]

SIX_GENE_PANEL = ["CDKN2A", "CDKN2B", "TP53BP1", "CDK4", "CD276", "MCL1"]
SIX_GENE_LOSS = {"CDKN2A", "CDKN2B", "TP53BP1"}
SIX_GENE_GAIN = {"CDK4", "CD276", "MCL1"}
MELANOMA_PRIORITY_GENES = ["BRAF", "NRAS", "NF1", "PTEN", "TP53", "KIT"]
SELECTED_HEATMAP_GENES = SIX_GENE_PANEL + MELANOMA_PRIORITY_GENES
DELETION_GENES = {"CDKN2A", "CDKN2B", "TP53BP1", "NF1", "PTEN", "TP53"}
AMP_GENES = {"CDK4", "CD276", "MCL1", "BRAF", "NRAS", "KIT"}

HEATMAP_TRANSITION_LABELS = {
    0: "None",
    1: "New amplification in metastasis",
    2: "Shared amplification",
    3: "New/shared uni-allelic loss in metastasis",
    4: "New functional biallelic loss in metastasis",
    5: "Shared functional biallelic loss",
}
HEATMAP_TRANSITION_COLORS = {
    0: "#ffffff",
    1: "#ffb3b3",
    2: "#b30000",
    3: "#9bd3ff",
    4: "#4ea8de",
    5: "#1d4e89",
}
ONCOCYCLE_STATE_LABELS = {
    "0->0": "OC- primary, OC- metastasis",
    "0->1": "OC- primary, OC+ metastasis",
    "1->1": "OC+ primary, OC+ metastasis",
    "1->0": "OC+ primary, OC- metastasis",
    "Missing": "Missing pair data",
}
ONCOCYCLE_STATE_COLORS = {
    "0->0": "#d9d9d9",
    "0->1": "#d7301f",
    "1->1": "#252525",
    "1->0": "#2171b5",
    "Missing": "#ffffff",
}

SNV_VARIANT_COLS = ["GOF_Oncogenic_Variants", "LOF_Oncogenic_One_Allele", "LOF_Oncogenic_Both_Alleles"]
CNV_GENE_PLOT_INCLUDE = sorted({
    "MCL1", "MDM2", "SETDB1", "PLCG2", "KIT", "EZH2", "NRAS", "EGFR", "BRAF",
    "CDK6", "CDK4", "MET", "NTRK1", "PLCG1", "MDM4", "PIK3CG", "PIK3C2B",
    "CCND1", "ERBB3", "HRAS", "MYC", "BCL6", "GLI1", "PPM1D", "CCND3",
    "CTNNB1", "PIK3CA", "IGF2R", "MAP2K1", "IGF1R", "E2F3", "BCL2", "NTRK3", "CD276",
    "ARID1B", "CDKN2A", "CDKN2B", "CDKN2C", "IFNGR1", "JAK2", "MTAP", "TP53BP1",
    "FAS", "TNFAIP3", "PTEN", "CASP8", "HLA-B", "BCL2L11", "PIK3R1", "TP53", "B2M",
})
CNV_GENE_HIGHLIGHTS = {
    "gain": ["CDK4", "MCL1", "CD276"],
    "loss": ["CDKN2A", "CDKN2B", "TP53BP1"],
}
CNV_REGION_LABEL_INCLUDE = ["12q12-q15", "1q21-q25", "15q21-q26", "9p21-p24", "15q11-q15"]
CNV_REGION_HIGHLIGHTS = {
    "gain": ["12q12-q15", "1q21-q25", "15q21-q26"],
    "loss": ["9p21-p24", "15q11-q15"],
}
MAX_MARKER_AREA = 1500
GRID_STEP = 0.2
CNV_MIN_TOTAL = 3
CNV_REGION_TARGET_N = 25
CNV_REGION_MIN_LOSS = 10
MAX_TERNARY_BAND_GAP = 2
COLOR_PRIMARY = "#395f83"
COLOR_MET = "#9878ae"
SHARED_LIGHT_TARGET = "#ffffff"
PANEL_BACKGROUND = "#ffffff"
GRID_COLOR = "#c3c7cf"
DISPLAY_NAME_MAP = {"CD276": "CD276"}
MARKER_MAP = {"snv": "o", "gain": "o", "loss": "s"}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Liberation Sans",
})


## Staged Inputs And Paired Cohort

The shared external clinical TSV now carries the Figure 3 pairing metadata (`panel_version`, `paired_met_code`, `paired_met_panel_version`, `met_site_simple`, `figure3_paired_primary`), so the notebook does not need any extra raw clinical cleanup.


In [ ]:

def build_required_inputs_table():
    rows = [
        ("Clinical Metadata", CLINICAL_METADATA_PATH, False, "external metadata", "paired primary cohort plus panel version and metastatic-site annotations"),
        ("filtered annotated VCFs", VCF_DIR / "**/filtered.*.annotated.vcf*", True, "downstream output", "TMB and MATH for the paired primary/met cohort"),
        ("combined ClinCNV somatic outputs", CLINCNV_RUN_DIR / "somatic/*/Annotated_ClinCNV_*.txt", True, "downstream of repo scripts", "FGA plus paired CNV state summaries"),
        ("OncoKB merged summary", PANEL_SEQ_ROOT / "new_bed_analysis/oncokb/all_oncokb_genes_annotated.txt", False, "downstream output", "curated SNV/CNV gene annotations for Figure 3E-G ternary plots"),
        ("OncoKB tables", ONCOKB_DIR / "*.oncokb.tsv", True, "downstream output", "oncogenic SNV and functional LOF summaries for paired plots"),
        ("v4 target BED", V4_BED_PATH, False, "downstream output", "panel-specific territory for TMB, MATH, and FGA on v4 samples"),
        ("v5 target BED", V5_BED_PATH, False, "downstream output", "panel-specific territory for TMB, MATH, and FGA on v5 samples"),
        ("v4 gene BED", V4_GENE_BED_PATH, False, "downstream output", "map oncogenic CNV genes to broad cytoband regions on v4"),
        ("v5 gene BED", V5_GENE_BED_PATH, False, "downstream output", "map oncogenic CNV genes to broad cytoband regions on v5"),
        ("cytoband reference", CYTOBAND_PATH, False, "reference metadata", "collapse oncogenic CNV genes into broad hg38 regions"),
    ]
    frame = pd.DataFrame(rows, columns=["input_family", "location", "is_glob", "category", "used_for"])
    frame["matches"] = frame.apply(lambda row: glob_count(row["location"]) if row["is_glob"] else path_count(row["location"]), axis=1)
    frame["present"] = frame["matches"] > 0
    return frame


def load_figure3_clinical_table(path=CLINICAL_METADATA_PATH):
    frame = pd.read_csv(path, sep="	")
    missing = [column for column in FIG3_REQUIRED_CLINICAL_COLUMNS if column not in frame.columns]
    if missing:
        raise ValueError(f"{path} is missing required Figure 3 clinical columns: {missing}")
    frame["Sample"] = frame["Sample"].astype(str).str.strip()
    if frame["Sample"].duplicated().any():
        duplicates = sorted(frame.loc[frame["Sample"].duplicated(), "Sample"].unique())
        raise ValueError(f"Clinical metadata contains duplicated Sample IDs: {duplicates}")
    frame = frame.set_index("Sample", drop=False).sort_index()
    for column in ["patient_id", "panel_version", "paired_met_code", "paired_met_panel_version", "met_site_simple"]:
        frame[column] = frame[column].fillna("").astype(str).str.strip()
    frame["figure3_paired_primary"] = pd.to_numeric(frame["figure3_paired_primary"], errors="coerce").fillna(0).astype(int)
    frame.loc[frame["met_site_simple"].eq(""), "met_site_simple"] = "Unknown"
    return frame


def load_figure3_pairs(clinical_df):
    pair_df = (
        clinical_df.reset_index(drop=True)
        .loc[
            lambda df: (df["figure3_paired_primary"] == 1)
            & df["paired_met_code"].ne("")
            & df["panel_version"].isin(["v4", "v5"])
            & df["paired_met_panel_version"].isin(["v4", "v5"])
        ,
            ["patient_id", "Sample", "panel_version", "paired_met_code", "paired_met_panel_version", "met_site_simple"]
        ]
        .rename(columns={
            "Sample": "primary_code",
            "panel_version": "primary_panel_version",
            "paired_met_code": "met_code",
            "paired_met_panel_version": "met_panel_version",
        })
        .drop_duplicates(subset=["patient_id", "primary_code", "met_code"])
        .copy()
    )
    pair_df["pair_id"] = pair_df["patient_id"] + ":" + pair_df["primary_code"] + "->" + pair_df["met_code"]
    pair_df["met_site_simple"] = pd.Categorical(pair_df["met_site_simple"], categories=PAIR_MET_SITE_ORDER, ordered=True)
    pair_df = pair_df.sort_values(["met_site_simple", "primary_code", "met_code"]).reset_index(drop=True)
    return pair_df


def read_clincnv_metadata(path):
    meta = {}
    with Path(path).open() as handle:
        for line in handle:
            if line.startswith("##"):
                item = line[2:].strip()
                if "=" in item:
                    key, value = item.split("=", 1)
                    meta[key.strip()] = value.strip()
                elif ":" in item:
                    key, value = item.split(":", 1)
                    meta[key.strip()] = value.strip()
            elif line.startswith("#"):
                break
    return meta


def estimated_fdr(path):
    meta = read_clincnv_metadata(path)
    for key, value in meta.items():
        if "estimated fdr" in key.lower():
            try:
                return float(value)
            except Exception:
                return np.inf
    return np.inf


def find_candidate_cna_files_for_tumor(sample_code):
    files = []
    for pair_dir in sorted(CLINCNV_SOMATIC_DIR.glob(f"{sample_code}-*")):
        pair_files = sorted(pair_dir.glob("Annotated_ClinCNV_*.txt")) or sorted(pair_dir.glob("CNAs_*.txt"))
        files.extend(pair_files)
    return files


def choose_best_cna_file(sample_code):
    files = find_candidate_cna_files_for_tumor(sample_code)
    if not files:
        return None
    ranked = sorted(((estimated_fdr(path), str(path), path) for path in files), key=lambda x: (x[0], x[1]))
    return ranked[0][2]


def read_clincnv_table(path):
    header = None
    rows = []
    with Path(path).open() as handle:
        for line in handle:
            if line.startswith('##'):
                continue
            if line.startswith('#'):
                header = line.lstrip('#').rstrip('\n').split('\t')
                continue
            if not line.strip() or header is None:
                continue
            parts = line.rstrip('\n').split('\t')
            if len(parts) < len(header):
                parts += [''] * (len(header) - len(parts))
            rows.append(parts[: len(header)])
    return pd.DataFrame(rows, columns=header)


def merge_intervals_0based(intervals):
    per_chr = defaultdict(list)
    for chrom, start, end in intervals:
        if start < end:
            per_chr[chrom].append((start, end))
    merged = {}
    for chrom, ivals in per_chr.items():
        ivals.sort()
        out = []
        for start, end in ivals:
            if not out or start > out[-1][1]:
                out.append([start, end])
            else:
                out[-1][1] = max(out[-1][1], end)
        merged[chrom] = [(start, end) for start, end in out]
    return merged


def load_targets_padded(path, pad=0):
    intervals = []
    with Path(path).open() as handle:
        for line in handle:
            if not line.strip() or line.startswith("#") or line.startswith("@"):
                continue
            parts = line.strip().split()
            if len(parts) < 3:
                continue
            chrom = parts[0]
            start = int(parts[1])
            end = int(parts[2])
            intervals.append((chrom, max(0, start - pad), end + pad))
    return merge_intervals_0based(intervals)


def territory_bp(merged):
    return sum(end - start for ivals in merged.values() for start, end in ivals)


def overlaps_0based(chrom, start0, end0, merged):
    intervals = merged.get(chrom, [])
    if not intervals:
        return False
    for tstart, tend in intervals:
        if tend <= start0:
            continue
        if tstart >= end0:
            break
        if max(tstart, start0) < min(tend, end0):
            return True
    return False


def clipped_target_overlaps(chrom, start0, end0, merged):
    out = []
    for tstart, tend in merged.get(chrom, []):
        if tend <= start0:
            continue
        if tstart >= end0:
            break
        clip_start = max(tstart, start0)
        clip_end = min(tend, end0)
        if clip_start < clip_end:
            out.append((clip_start, clip_end))
    return out


def union_len(intervals):
    if not intervals:
        return 0
    intervals = sorted(intervals)
    total = 0
    cur_start, cur_end = intervals[0]
    for start, end in intervals[1:]:
        if start <= cur_end:
            cur_end = max(cur_end, end)
        else:
            total += cur_end - cur_start
            cur_start, cur_end = start, end
    total += cur_end - cur_start
    return total


required_inputs = build_required_inputs_table()
display(Markdown("## Required staged inputs"))
display(required_inputs[["input_family", "category", "location", "used_for", "matches", "present"]])
missing_inputs = required_inputs.loc[~required_inputs["present"]].copy()
if not missing_inputs.empty:
    raise FileNotFoundError("Missing staged inputs for Figure 3 notebook:\n" + missing_inputs[["input_family", "location"]].to_string(index=False))

clinical_df = load_figure3_clinical_table()
pair_df = load_figure3_pairs(clinical_df)
if pair_df.empty:
    raise RuntimeError("No paired primary samples were available after loading the shared clinical metadata.")

pair_samples = sorted(set(pair_df["primary_code"]) | set(pair_df["met_code"]))
panel_version_by_sample = dict(zip(pair_df["primary_code"], pair_df["primary_panel_version"]))
panel_version_by_sample.update(dict(zip(pair_df["met_code"], pair_df["met_panel_version"])))
chosen_clincnv_file_by_sample = {sample: choose_best_cna_file(sample) for sample in pair_samples}

clinical_fields_used = pd.DataFrame([
    {"field": "Sample", "used_in": "paired primary join key", "source": "external clinical TSV"},
    {"field": "patient_id", "used_in": "pair labels and matched primary/met identifiers", "source": "external clinical TSV"},
    {"field": "panel_version", "used_in": "select v4 or v5 target territory for primary metrics", "source": "external clinical TSV"},
    {"field": "paired_met_code", "used_in": "matched metastasis sample code", "source": "external clinical TSV"},
    {"field": "paired_met_panel_version", "used_in": "select v4 or v5 target territory for metastasis metrics", "source": "external clinical TSV"},
    {"field": "met_site_simple", "used_in": "Figure 3D metastatic-site annotation row", "source": "external clinical TSV"},
    {"field": "figure3_paired_primary", "used_in": "restrict to the paired Figure 3 cohort", "source": "external clinical TSV"},
])
display(Markdown("## Minimal clinical fields actually used"))
display(clinical_fields_used)
display(pair_df.head(12))
print("Paired primaries in shared clinical metadata:", len(pair_df))
print("Primary/met samples represented across the paired cohort:", len(pair_samples))
print("Chosen ClinCNV files available for pair samples:", sum(path is not None for path in chosen_clincnv_file_by_sample.values()))
print("Metastatic-site counts:", pair_df["met_site_simple"].astype(str).value_counts(dropna=False).to_dict())


## Figure 3A-C: Paired Box Plots

These three panels reuse the paired primary/met cohort and rebuild TMB, MATH, and FGA directly from the staged VCF and ClinCNV files.


In [ ]:
from scipy.stats import wilcoxon

TMB_OVERLAP_PADDING_BP = 100
TMB_DENOM_PADDING_BP = 0
FGA_INTERVAL_PADDING_BP = 100
STRICT_SOMATIC = False
MATH_MIN_VARIANTS = 10
MATH_USE_SCALE_14826 = True

TMB_EFFECT_TO_CLASS = {
    "frameshift_variant": "Frame_Shift",
    "inframe_insertion": "In_Frame_Ins",
    "disruptive_inframe_insertion": "In_Frame_Ins",
    "conservative_inframe_insertion": "In_Frame_Ins",
    "inframe_deletion": "In_Frame_Del",
    "disruptive_inframe_deletion": "In_Frame_Del",
    "conservative_inframe_deletion": "In_Frame_Del",
    "missense_variant": "Missense_Mutation",
    "protein_altering_variant": "Missense_Mutation",
    "coding_sequence_variant": "Missense_Mutation",
    "stop_gained": "Nonsense_Mutation",
    "stop_lost": "Nonstop_Mutation",
    "start_lost": "Translation_Start_Site",
    "splice_acceptor_variant": "Splice_Site",
    "splice_donor_variant": "Splice_Site",
    "splice_region_variant": "Splice_Site",
}
TMB_NONSYN_CLASSES = {
    "Missense_Mutation", "Nonsense_Mutation", "Nonstop_Mutation", "Splice_Site",
    "Frame_Shift", "Frame_Shift_Ins", "Frame_Shift_Del", "In_Frame_Ins", "In_Frame_Del",
    "Translation_Start_Site",
}
TMB_SEVERITY_ORDER = [
    "frameshift_variant", "stop_gained", "stop_lost", "splice_acceptor_variant", "splice_donor_variant",
    "splice_region_variant", "start_lost", "inframe_deletion", "disruptive_inframe_deletion",
    "conservative_inframe_deletion", "inframe_insertion", "disruptive_inframe_insertion",
    "conservative_inframe_insertion", "missense_variant", "protein_altering_variant", "coding_sequence_variant",
]
TMB_SEVERITY_RANK = {effect: rank for rank, effect in enumerate(TMB_SEVERITY_ORDER)}

MATH_EFFECT_TO_CLASS = {
    "missense_variant": "Missense_Mutation",
    "protein_altering_variant": "Missense_Mutation",
    "coding_sequence_variant": "Missense_Mutation",
    "stop_gained": "Nonsense_Mutation",
    "stop_lost": "Nonstop_Mutation",
    "start_lost": "Translation_Start_Site",
    "splice_acceptor_variant": "Splice_Site",
    "splice_donor_variant": "Splice_Site",
    "splice_region_variant": "Splice_Site",
    "frameshift_variant": "Frameshift_Variant",
    "inframe_insertion": "Inframe_Indel",
    "inframe_deletion": "Inframe_Indel",
    "disruptive_inframe_insertion": "Inframe_Indel",
    "disruptive_inframe_deletion": "Inframe_Indel",
}
MATH_NONSYN_CLASSES = {
    "Missense_Mutation", "Nonsense_Mutation", "Nonstop_Mutation", "Splice_Site",
    "Translation_Start_Site", "Frameshift_Variant", "Inframe_Indel",
}
MATH_SEVERITY_ORDER = [
    "stop_gained", "frameshift_variant", "stop_lost",
    "splice_acceptor_variant", "splice_donor_variant", "splice_region_variant",
    "start_lost", "missense_variant", "protein_altering_variant",
    "inframe_insertion", "inframe_deletion", "coding_sequence_variant",
]
MATH_SEVERITY_RANK = {effect: rank for rank, effect in enumerate(MATH_SEVERITY_ORDER)}


def parse_ann_records(ann_value):
    records = []
    for record in str(ann_value or '').split(','):
        fields = record.split('|')
        while len(fields) < 18:
            fields.append('')
        records.append(fields)
    return records


def pick_best_effect_for_alt(records, severity_rank):
    best_record = None
    best_effect = None
    best_rank = 10 ** 9
    for record in records:
        for effect in (record[1] or '').split('&'):
            effect = effect.strip()
            rank = severity_rank.get(effect, 10 ** 9)
            if rank < best_rank:
                best_record = record
                best_effect = effect
                best_rank = rank
    return best_record, best_effect


def classify_tmb_variant(effect, ref, alt):
    base = TMB_EFFECT_TO_CLASS.get(str(effect).strip())
    if base is None:
        return None
    if base == 'Frame_Shift':
        if len(alt) > len(ref):
            return 'Frame_Shift_Ins'
        if len(alt) < len(ref):
            return 'Frame_Shift_Del'
    return base


def get_math_variant_class(info_dict, alt_allele):
    ann_raw = info_dict.get('ANN')
    if not ann_raw:
        return None
    ann_records = parse_ann_records(ann_raw)
    records_for_alt = [record for record in ann_records if (record[0] or '').split('/')[0] == alt_allele]
    if not records_for_alt:
        records_for_alt = ann_records
    best_effect = None
    best_rank = 10 ** 9
    for record in records_for_alt:
        for effect in (record[1] or '').split('&'):
            rank = MATH_SEVERITY_RANK.get(effect.strip(), 10 ** 9)
            if rank < best_rank:
                best_rank = rank
                best_effect = effect.strip()
    return MATH_EFFECT_TO_CLASS.get(best_effect)


def math_score_from_vafs(vafs, min_n=MATH_MIN_VARIANTS, use_scale_14826=MATH_USE_SCALE_14826):
    values = np.asarray(vafs, dtype=float)
    values = values[np.isfinite(values)]
    if values.size < min_n:
        return np.nan
    median_vaf = np.median(values)
    if median_vaf <= 0:
        return np.nan
    mad_vaf = np.median(np.abs(values - median_vaf))
    if use_scale_14826:
        mad_vaf = 1.4826 * mad_vaf
    return 100.0 * mad_vaf / median_vaf


def collapse_metric_df(metric_df, value_col):
    metric = metric_df[[value_col]].copy()
    metric[value_col] = pd.to_numeric(metric[value_col], errors='coerce')
    return metric.groupby(metric.index)[value_col].median().to_frame()


overlap_targets_by_panel = {
    'v4': load_targets_padded(V4_BED_PATH, pad=TMB_OVERLAP_PADDING_BP),
    'v5': load_targets_padded(V5_BED_PATH, pad=TMB_OVERLAP_PADDING_BP),
}
territory_targets_by_panel = {
    'v4': load_targets_padded(V4_BED_PATH, pad=TMB_DENOM_PADDING_BP),
    'v5': load_targets_padded(V5_BED_PATH, pad=TMB_DENOM_PADDING_BP),
}
fga_targets_by_panel = {
    'v4': load_targets_padded(V4_BED_PATH, pad=FGA_INTERVAL_PADDING_BP),
    'v5': load_targets_padded(V5_BED_PATH, pad=FGA_INTERVAL_PADDING_BP),
}
raw_panel_territory_mb = {panel: territory_bp(targets) / 1e6 for panel, targets in territory_targets_by_panel.items()}
panel_bp_fga = {panel: territory_bp(targets) for panel, targets in fga_targets_by_panel.items()}

all_vcf_paths = []
for pattern in VCF_GLOB:
    all_vcf_paths.extend(glob.glob(str(VCF_DIR / pattern), recursive=True))
all_vcf_paths = sorted(set(all_vcf_paths))

sample_to_vcf_paths = defaultdict(list)
for vcf_path in all_vcf_paths:
    sample = normalize_sample_id(sample_name_from_path(vcf_path))
    if sample in pair_samples:
        sample_to_vcf_paths[sample].append(Path(vcf_path))

# TMB exactly follows the Final_Fig_PriVsMet notebook: overlap filter on padded targets,
# denominator from the raw panel territory, then median-collapse duplicate sample rows.
tmb_rows = []
for sample in sorted(pair_samples):
    panel_version = panel_version_by_sample.get(sample, '')
    overlap_targets = overlap_targets_by_panel.get(panel_version)
    raw_denominator_mb = raw_panel_territory_mb.get(panel_version)
    for vcf_path in sample_to_vcf_paths.get(sample, []):
        nonsyn = 0
        with open_maybe_gzip(vcf_path) as handle:
            for line in handle:
                if not line or line.startswith('#'):
                    continue
                cols = line.rstrip("\n").split("\t")
                if len(cols) < 10:
                    continue
                chrom, pos, _id, ref, alts, qual, flt, info = cols[:8]
                if flt != 'PASS':
                    continue
                if not looks_somatic(info, strict=STRICT_SOMATIC):
                    continue
                try:
                    pos1 = int(pos)
                except Exception:
                    continue
                start0 = pos1 - 1
                end0 = start0 + len(ref)
                if overlap_targets is None or not overlaps_0based(chrom, start0, end0, overlap_targets):
                    continue
                info_dict = parse_info_dict(info)
                ann_raw = info_dict.get('ANN')
                if not ann_raw:
                    continue
                ann_records = parse_ann_records(ann_raw)
                for alt in alts.split(','):
                    matching_records = [record for record in ann_records if (record[0] or '').split('/')[0] == alt] or ann_records
                    best_record, best_effect = pick_best_effect_for_alt(matching_records, TMB_SEVERITY_RANK)
                    if not best_record or not best_effect:
                        continue
                    variant_class = classify_tmb_variant(best_effect, ref, alt)
                    if variant_class not in TMB_NONSYN_CLASSES:
                        continue
                    nonsyn += 1
        tmb_rows.append({
            'Sample': sample,
            'TMB_per_Mb': nonsyn / raw_denominator_mb if raw_denominator_mb and np.isfinite(raw_denominator_mb) else np.nan,
        })
metric_tmb_raw_df = pd.DataFrame(tmb_rows).set_index('Sample').sort_index() if tmb_rows else pd.DataFrame(columns=['TMB_per_Mb'])
metric_tmb_df = collapse_metric_df(metric_tmb_raw_df, 'TMB_per_Mb') if not metric_tmb_raw_df.empty else pd.DataFrame(columns=['TMB_per_Mb'])

# MATH follows the manuscript notebook too: last encountered VCF per sample and dp-aware filtering.
math_variant_records = {}
for vcf_path in all_vcf_paths:
    sample = normalize_sample_id(sample_name_from_path(vcf_path))
    if sample not in pair_samples:
        continue
    panel_version = panel_version_by_sample.get(sample, '')
    overlap_targets = overlap_targets_by_panel.get(panel_version)
    if overlap_targets is None:
        continue
    records = []
    with open_maybe_gzip(vcf_path) as handle:
        for line in handle:
            if not line or line.startswith('#'):
                continue
            cols = line.rstrip("\n").split("\t")
            if len(cols) < 10:
                continue
            chrom, pos, _id, ref, alts, qual, flt, info = cols[:8]
            if flt != 'PASS':
                continue
            if not looks_somatic(info, strict=STRICT_SOMATIC):
                continue
            try:
                pos1 = int(pos)
            except Exception:
                continue
            start0 = pos1 - 1
            end0 = start0 + len(ref)
            if not overlaps_0based(chrom, start0, end0, overlap_targets):
                continue
            info_dict = parse_info_dict(info)
            format_keys = cols[8].split(':')
            sample_vals = cols[9].split(':')
            for alt_index, alt in enumerate(alts.split(',')):
                variant_class = get_math_variant_class(info_dict, alt)
                if variant_class not in MATH_NONSYN_CLASSES:
                    continue
                vaf, dp = get_vaf_dp(format_keys, sample_vals, alt_index, info_dict=info_dict)
                if vaf is None:
                    continue
                records.append((float(vaf), None if dp is None else int(dp)))
    math_variant_records[sample] = records


def get_sample_math_vafs(sample, vaf_min=0.0, dp_min=0):
    values = []
    for vaf, dp in math_variant_records.get(sample, []):
        if vaf < vaf_min:
            continue
        if dp is None or dp < dp_min:
            continue
        values.append(vaf)
    return np.asarray(values, dtype=float)


math_rows = []
for sample in sorted(pair_samples):
    finite_vafs = get_sample_math_vafs(sample, vaf_min=0.0, dp_min=0)
    median_vaf = np.median(finite_vafs) if finite_vafs.size >= MATH_MIN_VARIANTS else np.nan
    mad_vaf = np.median(np.abs(finite_vafs - median_vaf)) if finite_vafs.size >= MATH_MIN_VARIANTS else np.nan
    math_rows.append({
        'Sample': sample,
        'MATH_num_variants_used': int(finite_vafs.size),
        'Median_VAF': median_vaf,
        'MAD_VAF': mad_vaf,
        'MATH': math_score_from_vafs(finite_vafs),
    })
metric_math_df = pd.DataFrame(math_rows).set_index('Sample').sort_index()
metric_vcf_df = metric_tmb_df.join(metric_math_df[['MATH', 'MATH_num_variants_used']], how='outer')

# FGA also matches Final_Fig_PriVsMet: compute every ClinCNV file, then median-collapse duplicates.
fga_rows = []
for clincnv_path in sorted(iter_clincnv_paths(CLINCNV_RUN_DIR)):
    sample = sample_from_clincnv_filename(clincnv_path).split('-')[0].strip()
    if sample not in pair_samples:
        continue
    panel_version = panel_version_by_sample.get(sample, '')
    merged_targets = fga_targets_by_panel.get(panel_version)
    covered_bp = panel_bp_fga.get(panel_version)
    if merged_targets is None or not covered_bp:
        continue
    by_chr = defaultdict(list)
    table = read_clincnv_table(clincnv_path)
    for row in table.itertuples(index=False):
        chrom = getattr(row, 'chr', '') if hasattr(row, 'chr') else getattr(row, '#chr', '')
        chrom = str(chrom)
        if chrom and not chrom.startswith('chr'):
            chrom = f'chr{chrom}'
        if chrom not in merged_targets:
            continue
        try:
            start = int(float(getattr(row, 'start')))
            end = int(float(getattr(row, 'end')))
        except Exception:
            continue
        if start >= end:
            continue
        state = str(getattr(row, 'state', '')).upper()
        if state not in {'AMP', 'DEL'}:
            continue
        by_chr[chrom].extend(clipped_target_overlaps(chrom, start, end, merged_targets))
    altered_bp = sum(union_len(by_chr.get(chrom, [])) for chrom in merged_targets)
    fga_rows.append({'Sample': sample, 'FGA_ampdel': altered_bp / covered_bp if covered_bp else np.nan})
metric_fga_raw_df = pd.DataFrame(fga_rows).set_index('Sample').sort_index() if fga_rows else pd.DataFrame(columns=['FGA_ampdel'])
metric_fga_df = collapse_metric_df(metric_fga_raw_df, 'FGA_ampdel') if not metric_fga_raw_df.empty else pd.DataFrame(columns=['FGA_ampdel'])


def build_paired_metric_df(metric_df, value_col):
    plot_df = pair_df[['patient_id', 'primary_code', 'met_code']].copy()
    series = pd.to_numeric(metric_df[value_col], errors='coerce')
    plot_df['primary_val'] = plot_df['primary_code'].map(series)
    plot_df['met_val'] = plot_df['met_code'].map(series)
    return plot_df.dropna(subset=['primary_val', 'met_val']).copy()


def plot_paired_metric_boxplot(ax, plot_df, ylabel):
    primary_vals = plot_df['primary_val'].to_numpy(dtype=float)
    met_vals = plot_df['met_val'].to_numpy(dtype=float)
    _, pval = wilcoxon(primary_vals, met_vals, alternative='two-sided', zero_method='wilcox') if len(plot_df) else (np.nan, np.nan)
    rng = np.random.default_rng(0)
    x_primary = rng.normal(1, 0.055, size=len(primary_vals))
    x_met = rng.normal(2, 0.055, size=len(met_vals))
    for idx in range(len(primary_vals)):
        ax.plot([x_primary[idx], x_met[idx]], [primary_vals[idx], met_vals[idx]], color='#d3d3d3', linewidth=1.2, zorder=1)
    ax.scatter(x_primary, primary_vals, s=22, facecolors='#bdbdbd', edgecolors='#666666', linewidths=0.7, alpha=0.9, zorder=2)
    ax.scatter(x_met, met_vals, s=22, facecolors='#bdbdbd', edgecolors='#666666', linewidths=0.7, alpha=0.9, zorder=2)
    box = ax.boxplot([primary_vals, met_vals], tick_labels=[f'Primary\n(n={len(primary_vals)})', f'Met\n(n={len(met_vals)})'], showfliers=False, widths=0.55, patch_artist=False)
    colors = {0: '#01afae', 1: '#b279de'}
    for idx, group in enumerate([0, 1]):
        color = colors[group]
        box['boxes'][idx].set(color=color, linewidth=1.8, zorder=3)
        box['medians'][idx].set(color=color, linewidth=2.2, zorder=3)
        box['whiskers'][2 * idx].set(color=color, linewidth=1.8, zorder=3)
        box['whiskers'][2 * idx + 1].set(color=color, linewidth=1.8, zorder=3)
        box['caps'][2 * idx].set(color=color, linewidth=1.8, zorder=3)
        box['caps'][2 * idx + 1].set(color=color, linewidth=1.8, zorder=3)
    ax.set_ylabel(ylabel)
    ax.text(0.03, 0.97, f'Wilcoxon p={pval:.2g}', transform=ax.transAxes, ha='left', va='top', fontsize=10)
    ax.tick_params(axis='x', labelsize=11)
    return pval


paired_tmb_df = build_paired_metric_df(metric_tmb_df, 'TMB_per_Mb')
paired_math_df = build_paired_metric_df(metric_math_df, 'MATH')
paired_fga_df = build_paired_metric_df(metric_fga_df, 'FGA_ampdel')

metric_summary = pd.DataFrame([
    {'metric': 'TMB_per_Mb', 'matched_pairs': len(paired_tmb_df)},
    {'metric': 'MATH', 'matched_pairs': len(paired_math_df)},
    {'metric': 'FGA_ampdel', 'matched_pairs': len(paired_fga_df)},
])
display(Markdown('## Figure 3A-C: paired primary-versus-metastasis box plots'))
display(metric_summary)

fig, axes = plt.subplots(1, 3, figsize=(8.8, 4.1), dpi=200)
p_tmb = plot_paired_metric_boxplot(axes[0], paired_tmb_df, 'TMB per Mb')
p_math = plot_paired_metric_boxplot(axes[1], paired_math_df, 'MATH score')
p_fga = plot_paired_metric_boxplot(axes[2], paired_fga_df, 'FGA')
fig.tight_layout()
plt.show()

paired_metric_plot_summary = pd.DataFrame([
    {'metric': 'TMB_per_Mb', 'matched_pairs': len(paired_tmb_df), 'wilcoxon_p': p_tmb, 'primary_median': paired_tmb_df['primary_val'].median(), 'met_median': paired_tmb_df['met_val'].median()},
    {'metric': 'MATH', 'matched_pairs': len(paired_math_df), 'wilcoxon_p': p_math, 'primary_median': paired_math_df['primary_val'].median(), 'met_median': paired_math_df['met_val'].median()},
    {'metric': 'FGA_ampdel', 'matched_pairs': len(paired_fga_df), 'wilcoxon_p': p_fga, 'primary_median': paired_fga_df['primary_val'].median(), 'met_median': paired_fga_df['met_val'].median()},
])
display(paired_metric_plot_summary)


## Figure 3D: Paired Heatmap

This panel summarizes six-gene OncoCycle transitions plus six-gene and melanoma-priority CNV state changes across the matched primary/met cohort.


In [ ]:
from matplotlib.patches import Rectangle

oncokb_pair_df = build_oncokb_summary(
    ONCOKB_DIR,
    VCF_DIR,
    min_vaf=0.0,
    strict_somatic=False,
    snv_only=False,
    include_samples=pair_samples,
).copy()
clincnv_pair_df = build_clincnv_summary(CLINCNV_RUN_DIR).copy()
clincnv_pair_df.index = clincnv_pair_df.index.map(lambda value: str(value).strip())
merged_pair_df = add_functional_biallelic_calls(clincnv_pair_df.join(oncokb_pair_df, how='left'))
merged_pair_df = merged_pair_df.loc[~merged_pair_df.index.duplicated(keep='first')].copy()

heatmap_pair_df = pair_df.loc[pair_df['primary_code'].isin(merged_pair_df.index) & pair_df['met_code'].isin(merged_pair_df.index)].copy()
if heatmap_pair_df.empty:
    raise RuntimeError('No complete primary/met pairs remained for the Figure 3D heatmap.')
heatmap_pair_df = heatmap_pair_df.sort_values(['met_site_simple', 'primary_code', 'met_code']).reset_index(drop=True)


def sample_gene_states(row):
    return {
        'amp': split_genes(row.get('genes_amp_str', '')),
        'gained': split_genes(row.get('genes_gained_str', '')),
        'hetdel': split_genes(row.get('genes_HETDEL_str', '')),
        'homdel': split_genes(row.get('genes_HOMDEL_str', '')),
        'loh': split_genes(row.get('genes_LOH_str', '')),
        'functional_biallelic': split_genes(row.get('functional_biallelic_genes', '')),
    }


def six_gene_oncocycle_value(row):
    states = sample_gene_states(row)
    has_loss = bool(SIX_GENE_LOSS & states['functional_biallelic'])
    has_gain = bool(SIX_GENE_GAIN & (states['amp'] | states['gained']))
    return int(has_loss or has_gain)


def six_gene_pair_oncocycle_transition(primary_row, met_row):
    p_onco = six_gene_oncocycle_value(primary_row)
    m_onco = six_gene_oncocycle_value(met_row)
    if p_onco and m_onco:
        return '1->1'
    if m_onco:
        return '0->1'
    if p_onco:
        return '1->0'
    return '0->0'


def classify_transition(gene, primary_row, met_row):
    p_states = sample_gene_states(primary_row)
    m_states = sample_gene_states(met_row)
    p_amp = gene in (p_states['amp'] | p_states['gained'])
    m_amp = gene in (m_states['amp'] | m_states['gained'])
    p_bi = gene in p_states['functional_biallelic']
    m_bi = gene in m_states['functional_biallelic']
    m_hetloss = gene in (m_states['hetdel'] | m_states['loh'])
    if gene in DELETION_GENES:
        if p_bi and m_bi:
            return 5
        if (not p_bi) and m_bi:
            return 4
        if m_hetloss:
            return 3
    if gene in AMP_GENES:
        if p_amp and m_amp:
            return 2
        if (not p_amp) and m_amp:
            return 1
    return 0

pair_ids = heatmap_pair_df['pair_id'].tolist()
met_types = pd.Series(heatmap_pair_df['met_site_simple'].astype(str).values, index=pair_ids)
heatmap_values = pd.DataFrame(index=SELECTED_HEATMAP_GENES, columns=pair_ids, dtype=int)
oncocycle_row = {}
for row in heatmap_pair_df.itertuples(index=False):
    primary_row = merged_pair_df.loc[row.primary_code]
    met_row = merged_pair_df.loc[row.met_code]
    if isinstance(primary_row, pd.DataFrame):
        primary_row = primary_row.iloc[-1]
    if isinstance(met_row, pd.DataFrame):
        met_row = met_row.iloc[-1]
    for gene in SELECTED_HEATMAP_GENES:
        heatmap_values.at[gene, row.pair_id] = classify_transition(gene, primary_row, met_row)
    oncocycle_row[row.pair_id] = six_gene_pair_oncocycle_transition(primary_row, met_row)
oncocycle_row = pd.Series(oncocycle_row).reindex(pair_ids).fillna('Missing')

met_type_colors = {
    'cutaneous / subcutaneous': '#66c2a5',
    'lymph node': '#fc8d62',
    'visceral': '#8da0cb',
    'Unknown': '#d9d9d9',
}
row_layout = [('MET_TYPE', 'Met site', 1.15), ('GAP', None, 0.75), ('ONCOCYCLE', 'Oncocycle', 1.15), ('GAP', None, 0.75)]
row_layout.extend(('GENE', gene, 1.0) for gene in SIX_GENE_PANEL)
row_layout.append(('GAP', None, 0.35))
row_layout.extend(('GENE', gene, 1.0) for gene in MELANOMA_PRIORITY_GENES)

total_h = float(sum(height for _, _, height in row_layout))
fig, ax = plt.subplots(figsize=(14.2, 5.6), dpi=250)
ax.set_xlim(0, len(pair_ids))
ax.set_ylim(0, total_h + 2.4)
ax.invert_yaxis()
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

y0 = 0.0
ytick_pos = []
ytick_lab = []
ytick_kind = []
section_boxes = []
current_section = None
section_start = None
for kind, payload, height in row_layout:
    if kind == 'GAP':
        if current_section is not None and section_start is not None:
            section_boxes.append((section_start, y0 - section_start))
            current_section = None
            section_start = None
        y0 += height
        continue
    if current_section is None:
        current_section = kind
        section_start = y0
    for xi, pair_id in enumerate(pair_ids):
        if kind == 'MET_TYPE':
            face = met_type_colors.get(str(met_types.loc[pair_id]), '#d9d9d9')
        elif kind == 'ONCOCYCLE':
            face = ONCOCYCLE_STATE_COLORS.get(oncocycle_row.loc[pair_id], '#ffffff')
        else:
            face = HEATMAP_TRANSITION_COLORS.get(int(heatmap_values.loc[payload, pair_id]), '#ffffff')
        ax.add_patch(Rectangle((xi, y0), 1, height, facecolor=face, edgecolor='#ffffff', linewidth=0.45))
    ytick_pos.append(y0 + height / 2)
    ytick_lab.append(payload)
    ytick_kind.append(kind)
    y0 += height
if current_section is not None and section_start is not None:
    section_boxes.append((section_start, y0 - section_start))
for y_start, height in section_boxes:
    ax.add_patch(Rectangle((0, y_start), len(pair_ids), height, facecolor='none', edgecolor='#111111', linewidth=1.5, clip_on=False))

met_boundaries = (met_types != met_types.shift()).to_numpy().nonzero()[0]
for boundary in met_boundaries[1:]:
    ax.axvline(boundary, color='#444444', linewidth=0.8, alpha=0.35, zorder=5)

ax.set_yticks(ytick_pos)
ax.set_yticklabels(ytick_lab, fontsize=12)
ax.tick_params(axis='y', length=0, pad=8)
for tick_label, kind in zip(ax.get_yticklabels(), ytick_kind):
    if kind == 'GENE':
        tick_label.set_fontstyle('italic')

legend_y = total_h + 0.35
legend_x = 0.0
legend_items = []
legend_items.extend([(met_type_colors[label], label) for label in ['cutaneous / subcutaneous', 'lymph node', 'visceral']])
legend_items.extend([(ONCOCYCLE_STATE_COLORS[key], ONCOCYCLE_STATE_LABELS[key]) for key in ['0->0','0->1','1->1','1->0']])
legend_items.extend([(HEATMAP_TRANSITION_COLORS[idx], HEATMAP_TRANSITION_LABELS[idx]) for idx in [1,2,3,4,5]])
for idx, (color, label) in enumerate(legend_items):
    row = idx // 4
    col = idx % 4
    x = 0.2 + col * (len(pair_ids) / 4.1)
    y = legend_y + row * 0.55
    ax.add_patch(Rectangle((x, y), 0.7, 0.32, facecolor=color, edgecolor='#666666', linewidth=0.3, clip_on=False))
    ax.text(x + 1.0, y + 0.16, label, va='center', ha='left', fontsize=9)

plt.tight_layout()
plt.show()
display(Markdown('## Figure 3D: paired primary-versus-metastasis OncoCycle/CNV heatmap'))
print('Heatmap pairs with complete ClinCNV + OncoKB summaries:', len(pair_ids))


## Figure 3E-G: Ternary Plots

These three panels summarize shared, primary-private, and metastasis-private oncogenic events across the matched cohort for SNVs, oncogenic CNV genes, and broad CNV regions.


In [ ]:
import matplotlib.colors as mcolors

ONCOKB_SUMMARY_PATH = PANEL_SEQ_ROOT / 'new_bed_analysis/oncokb/all_oncokb_genes_annotated.txt'
SNV_VARIANT_COLS = ['GOF_Oncogenic_Variants', 'LOF_Oncogenic_One_Allele', 'LOF_Oncogenic_Both_Alleles']
CNV_GAIN_COL = 'Oncogenic_GOF_AmpGain_Genes'
CNV_LOSS_COL = 'Functional_Oncogenic_Complete_LOF'
CNV_REGION_LOSS_COL = 'Functional_Oncogenic_Complete_LOF'
TARGET_GAIN_GENE_COLS = [
    'MCL1', 'MDM2', 'SETDB1', 'PLCG2', 'KIT', 'EZH2', 'NRAS', 'EGFR', 'BRAF',
    'CDK6', 'CDK4', 'MET', 'NTRK1', 'PLCG1', 'MDM4', 'PIK3CG', 'PIK3C2B',
    'CCND1', 'ERBB3', 'HRAS', 'MYC', 'BCL6', 'GLI1', 'PPM1D', 'CCND3',
    'CTNNB1', 'PIK3CA', 'IGF2R', 'MAP2K1', 'IGF1R', 'E2F3', 'BCL2', 'NTRK3', 'CD276',
]
TARGET_LOSS_GENE_COLS = [
    'ARID1B', 'CDKN2A', 'CDKN2B', 'CDKN2C', 'IFNGR1', 'JAK2', 'MTAP', 'TP53BP1',
    'FAS', 'TNFAIP3', 'PTEN', 'CASP8', 'HLA-B', 'BCL2L11', 'PIK3R1', 'TP53', 'B2M',
]
CNV_REGION_TARGET_LOOKUP = {
    'gain': ['12q12-q15', '1q21-q25', '15q21-q26'],
    'loss': ['9p21-p24', '15q11-q15'],
}


def parse_item_set(value, upper=False):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return set()
    text = str(value).strip()
    if not text:
        return set()
    items = {re.sub(r'\s+', ' ', token.strip()) for token in text.split(',') if token.strip()}
    return {item.upper() for item in items} if upper else items


def parse_gene_set(value):
    return parse_item_set(value, upper=True)


def normalize_item_list(values, upper=False):
    out = []
    for value in values or []:
        text = re.sub(r'\s+', ' ', str(value).strip())
        if not text:
            continue
        out.append(text.upper() if upper else text)
    return out


def extract_gene_from_variant_label(label):
    text = re.sub(r'\s+', ' ', str(label).strip())
    if not text:
        return None
    gene = text.split(' ', 1)[0].strip().upper()
    return gene or None


def load_gene_bed(path):
    df = pd.read_csv(path, sep='	', header=None, usecols=[0, 1, 2, 4], names=['chrom', 'start', 'end', 'gene'])
    df['chrom'] = df['chrom'].map(canon_chrom)
    df['start'] = pd.to_numeric(df['start'], errors='coerce').astype('Int64')
    df['end'] = pd.to_numeric(df['end'], errors='coerce').astype('Int64')
    df['gene'] = df['gene'].astype(str).str.strip().str.upper()
    df = df.dropna(subset=['start', 'end'])
    return (
        df.loc[df['chrom'].isin(AUTOSOMES) & df['gene'].ne('')]
        .groupby(['gene', 'chrom'], as_index=False)
        .agg(start=('start', 'min'), end=('end', 'max'))
    )


def load_broad_cytobands(path):
    raw = pd.read_csv(path, sep='	', header=None, names=['chrom', 'start', 'end', 'arm_band', 'stain'])
    raw['chrom'] = raw['chrom'].map(canon_chrom)
    raw['start'] = pd.to_numeric(raw['start'], errors='coerce').astype('Int64')
    raw['end'] = pd.to_numeric(raw['end'], errors='coerce').astype('Int64')
    raw = raw.dropna(subset=['start', 'end'])
    raw = raw.loc[raw['chrom'].isin(AUTOSOMES)].copy()
    def broad_cytoband_name(chrom, arm_band):
        match = re.match(r'^([pq])(\d+)', str(arm_band).strip(), flags=re.IGNORECASE)
        if not match:
            return None
        return f"{chrom}{match.group(1).lower()}{match.group(2)}"
    raw['cytoband'] = [broad_cytoband_name(chrom, arm_band) for chrom, arm_band in zip(raw['chrom'], raw['arm_band'])]
    raw = raw.loc[raw['cytoband'].notna()].copy()
    broad = raw.groupby(['chrom', 'cytoband'], as_index=False).agg(start=('start', 'min'), end=('end', 'max'))
    return broad.sort_values(['chrom', 'start', 'end', 'cytoband']).reset_index(drop=True)


def overlap_len(start_a, end_a, start_b, end_b):
    return max(0, min(end_a, end_b) - max(start_a, start_b))


def assign_genes_to_cytobands(gene_df, cytobands):
    rows = []
    by_chrom = {chrom: sub[['cytoband', 'start', 'end']].copy() for chrom, sub in cytobands.groupby('chrom', sort=False)}
    for row in gene_df.itertuples(index=False):
        sub = by_chrom.get(row.chrom)
        if sub is None or sub.empty:
            continue
        overlaps = sub.apply(lambda band: overlap_len(int(row.start), int(row.end), int(band.start), int(band.end)), axis=1)
        if int(overlaps.max()) <= 0:
            continue
        best = sub.iloc[int(overlaps.values.argmax())]
        rows.append({'gene': row.gene, 'cytoband': best['cytoband']})
    return pd.DataFrame(rows).drop_duplicates(subset=['gene'], keep='first')


def split_broad_cytoband(cytoband):
    match = re.match(r'^(?P<chrom>\d+)(?P<arm>[pq])(?P<band>\d+)$', str(cytoband).strip(), flags=re.IGNORECASE)
    if not match:
        return None
    return match.group('chrom'), match.group('arm').lower(), int(match.group('band'))


def format_region_label(chrom, arm, start_band, end_band):
    if start_band == end_band:
        return f'{chrom}{arm}{start_band}'
    return f'{chrom}{arm}{start_band}-{arm}{end_band}'


def build_cytoband_region_map(cytobands):
    rows = []
    for cytoband in sorted(set(cytobands)):
        parsed = split_broad_cytoband(cytoband)
        if not parsed:
            continue
        chrom, arm, band = parsed
        rows.append({'cytoband': cytoband, 'chrom': chrom, 'arm': arm, 'band': band})
    meta = pd.DataFrame(rows).sort_values(['chrom', 'arm', 'band', 'cytoband']).reset_index(drop=True)
    region_map = {}
    current_members = []
    current_key = None
    prev_band = None
    def flush():
        if not current_members:
            return
        chrom, arm = current_key
        label = format_region_label(chrom, arm, min(current_members), max(current_members))
        for band in current_members:
            region_map[f'{chrom}{arm}{band}'] = label
    for row in meta.itertuples(index=False):
        key = (row.chrom, row.arm)
        if current_key != key or prev_band is None or row.band > MAX_TERNARY_BAND_GAP + prev_band:
            flush()
            current_members = [row.band]
            current_key = key
        else:
            current_members.append(row.band)
        prev_band = row.band
    flush()
    return {row.cytoband: region_map.get(row.cytoband, row.cytoband) for row in meta.itertuples(index=False)}


def compute_ternary_counts(valid_pairs, sample_to_items, event_type):
    counts = {}
    for trio in valid_pairs.itertuples(index=False):
        primary_items = sample_to_items.get(trio.primary_code, set())
        met_items = sample_to_items.get(trio.met_code, set())
        for item in primary_items | met_items:
            key = (item, event_type)
            if key not in counts:
                counts[key] = {'item': item, 'event_type': event_type, 'primary_private': 0, 'met_private': 0, 'shared': 0, 'total': 0}
            if item in primary_items and item in met_items:
                counts[key]['shared'] += 1
            elif item in primary_items:
                counts[key]['primary_private'] += 1
            else:
                counts[key]['met_private'] += 1
            counts[key]['total'] += 1
    return pd.DataFrame(counts.values())


def compute_snv_gene_counts(onco_df, valid_pairs):
    gof_sets = {
        sample: {
            extract_gene_from_variant_label(item)
            for item in parse_item_set(row.get('GOF_Oncogenic_Variants', ''), upper=False)
            if extract_gene_from_variant_label(item)
        }
        for sample, row in onco_df.iterrows()
    }
    lof_sets = {
        sample: {
            extract_gene_from_variant_label(item)
            for col in ['LOF_Oncogenic_One_Allele', 'LOF_Oncogenic_Both_Alleles']
            for item in parse_item_set(row.get(col, ''), upper=False)
            if extract_gene_from_variant_label(item)
        }
        for sample, row in onco_df.iterrows()
    }
    snv_df = pd.concat([
        compute_ternary_counts(valid_pairs, gof_sets, 'snv'),
        compute_ternary_counts(valid_pairs, lof_sets, 'snv'),
    ], ignore_index=True)
    if snv_df.empty:
        return snv_df
    return (
        snv_df.groupby('item', as_index=False)[['primary_private', 'met_private', 'shared', 'total']]
        .sum()
        .assign(event_type='snv')
        .sort_values(['total', 'shared', 'item'], ascending=[False, False, True])
        .reset_index(drop=True)
    )


def compute_cnv_gene_counts(onco_df, valid_pairs):
    gain_sets = {sample: parse_gene_set(row.get(CNV_GAIN_COL, '')) for sample, row in onco_df.iterrows()}
    loss_sets = {sample: parse_gene_set(row.get(CNV_LOSS_COL, '')) for sample, row in onco_df.iterrows()}
    return pd.concat([
        compute_ternary_counts(valid_pairs, gain_sets, 'gain'),
        compute_ternary_counts(valid_pairs, loss_sets, 'loss'),
    ], ignore_index=True)


def genes_to_region_set(genes, gene_to_region):
    return {gene_to_region[gene] for gene in genes if gene in gene_to_region and gene_to_region[gene]}


def compute_cnv_region_counts(onco_df, valid_pairs, gene_to_region):
    gain_sets = {sample: genes_to_region_set(parse_gene_set(row.get(CNV_GAIN_COL, '')), gene_to_region) for sample, row in onco_df.iterrows()}
    loss_sets = {sample: genes_to_region_set(parse_gene_set(row.get(CNV_REGION_LOSS_COL, '')), gene_to_region) for sample, row in onco_df.iterrows()}
    return pd.concat([
        compute_ternary_counts(valid_pairs, gain_sets, 'gain'),
        compute_ternary_counts(valid_pairs, loss_sets, 'loss'),
    ], ignore_index=True)


def include_rows_by_event_targets(df, target_lookup, upper=False):
    rows = []
    for event_type, items in (target_lookup or {}).items():
        if not items:
            continue
        include = set(normalize_item_list(items, upper=upper))
        series = df['item'].astype(str).str.upper() if upper else df['item'].astype(str)
        rows.append(df.loc[(df['event_type'] == event_type) & series.isin(include)].copy())
    if not rows:
        return df.iloc[0:0].copy()
    return pd.concat(rows, ignore_index=True).drop_duplicates(subset=['item', 'event_type'], keep='first')


def filter_counts_by_event_targets(df, target_lookup, upper=False):
    out = include_rows_by_event_targets(df, target_lookup, upper=upper)
    return out.sort_values(['total', 'shared', 'item'], ascending=[False, False, True]).reset_index(drop=True)


def build_balanced_event_subset(df, total_n, min_loss, target_lookup=None, upper=False):
    df = df.copy()
    if df.empty:
        return df
    forced = include_rows_by_event_targets(df, target_lookup, upper=upper)
    gain_pool = df.loc[df['event_type'] == 'gain'].sort_values(['total', 'shared', 'item'], ascending=[False, False, True])
    loss_pool = df.loc[df['event_type'] == 'loss'].sort_values(['total', 'shared', 'item'], ascending=[False, False, True])
    selected_losses = forced.loc[forced['event_type'] == 'loss'].copy()
    if len(selected_losses) < min_loss:
        remaining_losses = loss_pool.merge(selected_losses[['item', 'event_type']].drop_duplicates(), on=['item', 'event_type'], how='left', indicator=True)
        remaining_losses = remaining_losses.loc[remaining_losses['_merge'] == 'left_only'].drop(columns=['_merge'])
        selected_losses = pd.concat([selected_losses, remaining_losses.head(min_loss - len(selected_losses))], ignore_index=True)
    selected = pd.concat([forced.loc[forced['event_type'] == 'gain'].copy(), selected_losses], ignore_index=True)
    selected = selected.drop_duplicates(subset=['item', 'event_type'], keep='first')
    remaining_slots = max(0, total_n - len(selected))
    if remaining_slots:
        remaining_gains = gain_pool.merge(selected[['item', 'event_type']].drop_duplicates(), on=['item', 'event_type'], how='left', indicator=True)
        remaining_gains = remaining_gains.loc[remaining_gains['_merge'] == 'left_only'].drop(columns=['_merge'])
        selected = pd.concat([selected, remaining_gains.head(remaining_slots)], ignore_index=True)
    if len(selected) < total_n:
        remaining_any = df.sort_values(['total', 'shared', 'item'], ascending=[False, False, True]).merge(selected[['item', 'event_type']].drop_duplicates(), on=['item', 'event_type'], how='left', indicator=True)
        remaining_any = remaining_any.loc[remaining_any['_merge'] == 'left_only'].drop(columns=['_merge'])
        selected = pd.concat([selected, remaining_any.head(total_n - len(selected))], ignore_index=True)
    return selected.drop_duplicates(subset=['item', 'event_type'], keep='first').head(total_n).reset_index(drop=True)


def add_plot_columns(plot_df):
    plot_df = plot_df.copy()
    plot_df['primary_frac'] = plot_df['primary_private'] / plot_df['total']
    plot_df['met_frac'] = plot_df['met_private'] / plot_df['total']
    plot_df['shared_frac'] = plot_df['shared'] / plot_df['total']
    plot_df['x_plot'] = plot_df['met_frac'] + 0.5 * plot_df['shared_frac']
    plot_df['y_plot'] = (np.sqrt(3) / 2) * plot_df['shared_frac']
    plot_df['marker_area'] = (plot_df['total'] / plot_df['total'].max()) * MAX_MARKER_AREA
    return plot_df


def blend_rgb(rgb1, rgb2, weight):
    return tuple((1 - weight) * a + weight * b for a, b in zip(rgb1, rgb2))


def lighten_toward_white(rgb, amount):
    white = np.array([1.0, 1.0, 1.0])
    base = np.array(rgb)
    return tuple((1 - amount) * base + amount * white)


def get_ternary_color(row):
    total_private = row['primary_private'] + row['met_private']
    met_weight = row['met_private'] / total_private if total_private else 0.5
    base = blend_rgb(mcolors.to_rgb(COLOR_PRIMARY), mcolors.to_rgb(COLOR_MET), met_weight)
    return lighten_toward_white(base, row['shared_frac'] * 0.85)


def select_labels(plot_df, max_labels, always_include=None):
    always_include = set(always_include or [])
    label_df = plot_df.copy()
    label_df['priority'] = label_df['item'].isin(always_include).astype(int)
    label_df = label_df.sort_values(['priority', 'total', 'shared', 'item'], ascending=[False, False, False, True])
    return label_df.head(max_labels).copy()


def draw_triangle(ax, triangle_height):
    ax.plot([0, 1, 0.5, 0], [0, 0, triangle_height, 0], color='black', linewidth=2.0, zorder=2)
    for step in np.arange(GRID_STEP, 1.0, GRID_STEP):
        ax.plot([0.5 * step, 1 - 0.5 * step], [triangle_height * step, triangle_height * step], color=GRID_COLOR, linewidth=0.8, alpha=0.55, zorder=1)
        ax.plot([1 - step, 0.5 * (1 - step)], [0, triangle_height * (1 - step)], color=GRID_COLOR, linewidth=0.8, alpha=0.55, zorder=1)
        ax.plot([step, 0.5 + 0.5 * step], [0, triangle_height * (1 - step)], color=GRID_COLOR, linewidth=0.8, alpha=0.55, zorder=1)


def add_axis_labels(ax, triangle_height):
    ax.text(-0.06, -0.045, 'Primary tumor-private', ha='left', va='top', fontsize=12, color=COLOR_PRIMARY, fontweight='bold')
    ax.text(1.06, -0.045, 'Metastasis-private', ha='right', va='top', fontsize=12, color=COLOR_MET, fontweight='bold')
    ax.text(0.5, triangle_height + 0.05, 'Shared', ha='center', va='bottom', fontsize=12, color='#333333', fontweight='bold')


def plot_panel(ax, plot_df, title, max_labels, always_include=None):
    triangle_height = np.sqrt(3) / 2
    draw_triangle(ax, triangle_height)
    if plot_df.empty:
        ax.text(0.5, triangle_height / 2, 'No events', ha='center', va='center', fontsize=12)
        ax.axis('off')
        return
    plot_df = add_plot_columns(plot_df)
    plot_df['color_rgb'] = plot_df.apply(get_ternary_color, axis=1)
    for event_type, group in plot_df.groupby('event_type', sort=False):
        ax.scatter(group['x_plot'], group['y_plot'], s=group['marker_area'], c=group['color_rgb'].tolist(), marker=MARKER_MAP.get(event_type, 'o'), edgecolors='black', linewidths=0.8, alpha=1.0, zorder=5, clip_on=False)
    label_df = select_labels(plot_df, max_labels=max_labels, always_include=always_include)
    for idx, row in label_df.reset_index(drop=True).iterrows():
        angle = 2 * np.pi * (idx / max(len(label_df), 1))
        dx = 0.055 * np.cos(angle)
        dy = 0.045 * np.sin(angle)
        x_text = row['x_plot'] + dx
        y_text = row['y_plot'] + dy
        face = '#ffe6df' if row['event_type'] == 'gain' else '#e4f0ff' if row['event_type'] == 'loss' else '#f7f7f7'
        edge = '#a63d2e' if row['event_type'] == 'gain' else '#2f5d8a' if row['event_type'] == 'loss' else '#666666'
        ax.annotate('', xy=(row['x_plot'], row['y_plot']), xytext=(x_text, y_text), arrowprops=dict(arrowstyle='-', lw=0.9, color=row['color_rgb'], shrinkA=5, shrinkB=5), zorder=4)
        ax.text(x_text, y_text, row['item'], fontsize=8.5, fontstyle='italic' if row['event_type'] == 'snv' else 'normal', ha='center', va='center', zorder=7, bbox=dict(boxstyle='round,pad=0.22', facecolor=face, edgecolor=edge, linewidth=0.8, alpha=0.98))
    add_axis_labels(ax, triangle_height)
    ax.set_title(title, pad=18)
    ax.set_xlim(-0.11, 1.12)
    ax.set_ylim(-0.08, triangle_height + 0.10)
    ax.set_aspect('equal')
    ax.axis('off')


expected_samples = pd.Index(pd.unique(pd.concat([pair_df['primary_code'], pair_df['met_code']], ignore_index=True).dropna()), name='Sample')
onco_df = pd.read_csv(ONCOKB_SUMMARY_PATH, sep='	', dtype=str).fillna('')
onco_df['Sample'] = onco_df['Sample'].astype(str).str.strip()
onco_df = onco_df.drop_duplicates(subset='Sample', keep='first').set_index('Sample')
onco_df = onco_df.reindex(expected_samples).fillna('')
strict_onco_df = build_oncokb_summary(ONCOKB_DIR, VCF_DIR, min_vaf=0.0, strict_somatic=False, snv_only=False, include_samples=expected_samples).copy()
strict_onco_df.index = strict_onco_df.index.map(lambda value: str(value).strip())
for column in ['GOF_Oncogenic_Variants', 'LOF_Oncogenic_One_Allele', 'LOF_Oncogenic_Both_Alleles', 'GOF_Genes', 'LOF_OneAllele_Genes', 'LOF_Biallelic_Genes']:
    if column in strict_onco_df.columns:
        onco_df[column] = strict_onco_df[column].reindex(expected_samples).fillna('')
if 'Variant_Richness' in strict_onco_df.columns:
    onco_df['Variant_Richness'] = pd.to_numeric(strict_onco_df['Variant_Richness'].reindex(expected_samples), errors='coerce').fillna(0).astype(int)

ternary_pairs = pair_df.loc[pair_df['primary_code'].isin(onco_df.index) & pair_df['met_code'].isin(onco_df.index)].copy()

snv_counts_df = compute_snv_gene_counts(onco_df, ternary_pairs)
snv_plot_df = snv_counts_df.head(30).copy()

cnv_gene_counts_df = compute_cnv_gene_counts(onco_df, ternary_pairs)
cnv_gene_target_lookup = {'gain': TARGET_GAIN_GENE_COLS, 'loss': TARGET_LOSS_GENE_COLS}
cnv_gene_plot_df = filter_counts_by_event_targets(cnv_gene_counts_df, cnv_gene_target_lookup, upper=True)

gene_df = pd.concat([load_gene_bed(V4_GENE_BED_PATH), load_gene_bed(V5_GENE_BED_PATH)], ignore_index=True)
gene_df = gene_df.groupby(['gene', 'chrom'], as_index=False).agg(start=('start', 'min'), end=('end', 'max'))
cytobands = load_broad_cytobands(CYTOBAND_PATH)
gene_cytobands = assign_genes_to_cytobands(gene_df, cytobands)
cytoband_to_region = build_cytoband_region_map(gene_cytobands['cytoband'])
gene_to_region = dict(zip(gene_cytobands['gene'], gene_cytobands['cytoband'].map(cytoband_to_region)))
cnv_region_counts_df = compute_cnv_region_counts(onco_df, ternary_pairs, gene_to_region)
cnv_region_plot_df = build_balanced_event_subset(cnv_region_counts_df, CNV_REGION_TARGET_N, CNV_REGION_MIN_LOSS, target_lookup=CNV_REGION_TARGET_LOOKUP, upper=False)

display(Markdown('## Figure 3E-G: paired primary-versus-metastasis ternary plots'))
print('Ternary pairs with curated OncoKB summary rows:', len(ternary_pairs))
print('SNV labels plotted:', len(snv_plot_df))
print('CNV-by-gene labels plotted:', len(cnv_gene_plot_df))
print('CNV-by-region labels plotted:', len(cnv_region_plot_df))

fig, axes = plt.subplots(1, 3, figsize=(18.5, 6.4), dpi=220)
plot_panel(axes[0], snv_plot_df, 'Shared', max_labels=len(snv_plot_df), always_include=[])
plot_panel(axes[1], cnv_gene_plot_df, 'Shared', max_labels=len(cnv_gene_plot_df), always_include=sum(CNV_GENE_HIGHLIGHTS.values(), []))
plot_panel(axes[2], cnv_region_plot_df, 'Shared', max_labels=len(cnv_region_plot_df), always_include=sum(CNV_REGION_HIGHLIGHTS.values(), []))
fig.tight_layout()
plt.show()
